In [2]:
"""
Gene-to-m/z Spatial Pattern Matching Pipeline V8
=================================================

KEY ENHANCEMENTS (V8):
- Pre-compute region labels using K-means or Leiden clustering at the start
- Region labels are computed ONCE per sample and reused for all features
- Region-to-region similarity comparison
- Global + regional matching scores

PARAMETERS:
- RNA (Visium): ~55 μm pixel size, hexagonal grid, 6 neighbors
- MSI: 60 μm pixel size, Cartesian grid, 8 neighbors
- RNA needs 180° rotation to align with MSI

Author: V8 - Pre-computed clustering + region-level matching
"""

import numpy as np
import scanpy as sc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
from scipy.stats import pearsonr, spearmanr
from scipy.interpolate import griddata
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from typing import List, Dict, Tuple, Optional, Union
import pandas as pd
import os
import warnings
from collections import defaultdict
from dataclasses import dataclass, field
import pickle

# Optional: Leiden clustering (if available)
try:
    import igraph as ig
    import leidenalg
    LEIDEN_AVAILABLE = True
except ImportError:
    LEIDEN_AVAILABLE = False
    print("Note: leidenalg not available, using K-means only")

warnings.filterwarnings('ignore')


# =============================================================================
# DATA CONFIGURATION
# =============================================================================

# Pixel sizes
RNA_PIXEL_SIZE = 55  # μm (Visium)
MSI_PIXEL_SIZE = 60  # μm

# Number of clusters/regions
N_CLUSTERS = 3

# Clustering method: 'kmeans' or 'leiden'
CLUSTERING_METHOD = 'leiden' #'kmeans'  # Change to 'leiden' if preferred

MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common/"
MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered_common.h5ad", "halfbrain_yc_2_filtered_common.h5ad",
    "halfbrain_yc_3_filtered_common.h5ad", "halfbrain_yc_4_filtered_common.h5ad",
    "halfbrain_yad_1_filtered_common.h5ad", "halfbrain_yad_2_filtered_common.h5ad",
    "halfbrain_yad_3_filtered_common.h5ad", "halfbrain_yad_4_filtered_common.h5ad",
    "halfbrain_ac_1_filtered_common.h5ad", "halfbrain_ac_2_filtered_common.h5ad",
    "halfbrain_ac_3_filtered_common.h5ad", "halfbrain_ac_4_filtered_common.h5ad",
    "halfbrain_aad_1_filtered_common.h5ad", "halfbrain_aad_2_filtered_common.h5ad",
    "halfbrain_aad_3_filtered_common.h5ad", "halfbrain_aad_4_filtered_common.h5ad"
]
MSI_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

RNA_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/genes_top_800/"
RNA_SAMPLE_FILES = [
    "YC_1.h5ad", "YC_2.h5ad", "YC_3.h5ad", "YC_4.h5ad",
    "YAD_1.h5ad", "YAD_2.h5ad", "YAD_3.h5ad", "YAD_4.h5ad",
    "AC_1.h5ad", "AC_2.h5ad", "AC_3.h5ad", "AC_4.h5ad",
    "AAD_1.h5ad", "AAD_2.h5ad", "AAD_3.h5ad", "AAD_4.h5ad"
]
RNA_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

AAD_TARGET_GENES = [
    'Thy1', 'App', 'Apoe', 'Mapt'
]


# =============================================================================
# CLUSTERING PIPELINE (SCANPY-STYLE)
# =============================================================================


def prepare_clustering_features(
    X: np.ndarray,
    n_pca_components: int = 30,
    verbose: bool = False
) -> np.ndarray:
    """
    Prepare features for clustering using scanpy-style pipeline.
    
    Pipeline:
    1. StandardScaler on intensity matrix
    2. PCA reduction
    
    Args:
        X: (N, M) array of intensities (cells × features)
        n_pca_components: Number of PCA components
        verbose: Print info
    
    Returns:
        (N, n_components) array of PCA-transformed features
    """
    n_points, n_features = X.shape
    n_components = min(n_pca_components, n_points - 1, n_features)
    
    if n_components < 1:
        n_components = 1
    
    # Step 1: Scale
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Step 2: PCA
    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_scaled)
    
    if verbose:
        print(f"    Features: {X.shape} -> scaled -> PCA({n_components}) -> {X_pca.shape}")
        print(f"    Variance explained: {pca.explained_variance_ratio_.sum():.2%}")
    
    return X_pca


def kmeans_cluster_regions(
    X: np.ndarray,
    coords: np.ndarray = None,
    n_neighbors: int = 15,
    n_clusters: int = 2,
    n_pca_components: int = 30,
    seed: int = 42,
    verbose: bool = False
) -> np.ndarray:
    """
    Apply K-means clustering using scanpy-style pipeline on intensity data.
    
    Pipeline:
    1. StandardScaler on intensities
    2. PCA reduction
    3. K-means clustering
    
    Args:
        X: (N, M) array of intensities (cells × features)
        coords: Not used (kept for API compatibility)
        n_neighbors: Not used (kept for API compatibility)
        n_clusters: Target number of clusters
        n_pca_components: Number of PCA components
        seed: Random seed
        verbose: Print debugging information
    
    Returns:
        (N,) array of cluster labels (0 to n_clusters-1)
    """
    n_points = X.shape[0]
    
    if n_points < n_clusters:
        if verbose:
            print(f"    Warning: Not enough points ({n_points}) for {n_clusters} clusters")
        return np.zeros(n_points, dtype=int)
    
    # Prepare features (scale + PCA on intensities)
    X_pca = prepare_clustering_features(X, n_pca_components, verbose)
    
    # K-means clustering
    kmeans = KMeans(
        n_clusters=n_clusters,
        random_state=seed,
        n_init=10,
        max_iter=300
    )
    labels = kmeans.fit_predict(X_pca)
    
    if verbose:
        unique, counts = np.unique(labels, return_counts=True)
        size_str = ", ".join([f"R{u}={c}" for u, c in zip(unique, counts)])
        print(f"    K-means: {n_points} points -> {n_clusters} clusters: {size_str}")
    
    return labels


def leiden_cluster_regions(
    X: np.ndarray,
    coords: np.ndarray = None,
    n_neighbors: int = 15,
    n_clusters: int = 2,
    n_pca_components: int = 30,
    resolution: float = 1.0,
    n_iterations: int = 2,
    seed: int = 42,
    verbose: bool = False
) -> np.ndarray:
    """
    Apply Leiden clustering using scanpy-style pipeline on intensity data.
    
    Pipeline:
    1. StandardScaler on intensities
    2. PCA reduction  
    3. KNN graph construction
    4. Leiden clustering
    
    Args:
        X: (N, M) array of intensities (cells × features)
        coords: Coordinates (used for post-processing only)
        n_neighbors: Number of neighbors for KNN graph
        n_clusters: Target number of clusters (used for resolution tuning)
        n_pca_components: Number of PCA components
        resolution: Leiden resolution parameter
        n_iterations: Number of Leiden iterations
        seed: Random seed
        verbose: Print debugging information
    
    Returns:
        (N,) array of cluster labels
    """
    if not LEIDEN_AVAILABLE:
        if verbose:
            print("    Leiden not available, falling back to K-means")
        return kmeans_cluster_regions(X, coords, n_neighbors, n_clusters, 
                                      n_pca_components, seed, verbose)
    
    n_points = X.shape[0]
    
    if n_points < n_clusters:
        if verbose:
            print(f"    Warning: Not enough points ({n_points}) for {n_clusters} clusters")
        return np.zeros(n_points, dtype=int)
    
    # Prepare features (scale + PCA on intensities)
    X_pca = prepare_clustering_features(X, n_pca_components, verbose)
    
    # Build KNN graph
    k = min(n_neighbors, n_points - 1)
    nn = NearestNeighbors(n_neighbors=k)
    nn.fit(X_pca)
    distances, indices = nn.kneighbors(X_pca)
    
    # Build igraph graph
    edges = []
    for i in range(indices.shape[0]):
        for j in indices[i][1:]:  # skip itself
            edges.append((i, j))
    
    g = ig.Graph(edges=edges, directed=False).simplify()
    
    if verbose:
        print(f"    Leiden: {n_points} nodes, {g.ecount()} edges, k={k}")
    
    # Leiden clustering with resolution search to get target clusters
    best_labels = None
    best_diff = float('inf')
    best_res = 1.0
    
    # Try different resolutions to get target number of clusters
    for res in [0.1, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0]:
        partition = leidenalg.find_partition(
            g,
            leidenalg.RBConfigurationVertexPartition,
            resolution_parameter=res,
            n_iterations=n_iterations,
            seed=seed
        )
        
        labels = np.array(partition.membership)
        n_found = len(np.unique(labels))
        diff = abs(n_found - n_clusters)
        
        if diff < best_diff:
            best_diff = diff
            best_labels = labels
            best_res = res
        
        if n_found == n_clusters:
            break
    
    labels = best_labels
    
    # Post-process to get exactly n_clusters (use coords if available)
    ref_coords = coords if coords is not None else X_pca[:, :2]
    
    # Merge if too many clusters
    while len(np.unique(labels)) > n_clusters:
        counts = {l: np.sum(labels == l) for l in np.unique(labels)}
        smallest = min(counts, key=counts.get)
        centroids = {l: ref_coords[labels == l].mean(axis=0) for l in np.unique(labels) if l != smallest}
        smallest_centroid = ref_coords[labels == smallest].mean(axis=0)
        nearest = min(centroids, key=lambda l: np.linalg.norm(centroids[l] - smallest_centroid))
        labels[labels == smallest] = nearest
    
    # Split if too few clusters
    while len(np.unique(labels)) < n_clusters:
        counts = {l: np.sum(labels == l) for l in np.unique(labels)}
        largest = max(counts, key=counts.get)
        largest_mask = labels == largest
        sub_labels = spatial_bisection(ref_coords[largest_mask], None)
        next_label = labels.max() + 1
        labels[largest_mask] = np.where(sub_labels == 0, largest, next_label)
    
    # Relabel to 0, 1, ..., n_clusters-1
    unique = np.unique(labels)
    label_map = {old: new for new, old in enumerate(unique)}
    labels = np.array([label_map[l] for l in labels])
    
    if verbose:
        unique, counts = np.unique(labels, return_counts=True)
        size_str = ", ".join([f"R{u}={c}" for u, c in zip(unique, counts)])
        print(f"    Leiden (res={best_res:.1f}): {size_str}")
    
    return labels


def cluster_regions(
    X: np.ndarray,
    coords: np.ndarray = None,
    n_clusters: int = 2,
    method: str = 'kmeans',
    n_neighbors: int = 15,
    n_pca_components: int = 30,
    seed: int = 42,
    verbose: bool = False
) -> np.ndarray:
    """
    Unified function to cluster spatial regions using specified method.
    
    This follows the scanpy-style pipeline:
    1. StandardScaler on intensities
    2. PCA
    3. KMeans or Leiden
    
    Args:
        X: (N, M) array of intensities (cells × features)
        coords: (N, 2) array of spatial coordinates (for post-processing)
        n_clusters: Number of clusters to create
        method: 'kmeans' or 'leiden'
        n_neighbors: Number of neighbors (for Leiden KNN graph)
        n_pca_components: Number of PCA components
        seed: Random seed
        verbose: Print debugging information
    
    Returns:
        (N,) array of cluster labels (0 to n_clusters-1)
    """
    if method.lower() == 'leiden':
        return leiden_cluster_regions(
            X=X,
            coords=coords,
            n_neighbors=n_neighbors,
            n_clusters=n_clusters,
            n_pca_components=n_pca_components,
            seed=seed,
            verbose=verbose
        )
    else:  # Default to kmeans
        return kmeans_cluster_regions(
            X=X,
            coords=coords,
            n_neighbors=n_neighbors,
            n_clusters=n_clusters,
            n_pca_components=n_pca_components,
            seed=seed,
            verbose=verbose
        )


def split_to_n_clusters(
    coords: np.ndarray,
    values: np.ndarray,
    labels: np.ndarray,
    n_clusters: int,
    verbose: bool = False
) -> np.ndarray:
    """
    Split clusters until we have exactly n_clusters.
    Recursively splits the largest cluster.
    """
    current_labels = labels.copy()
    next_label = current_labels.max() + 1
    
    while len(np.unique(current_labels)) < n_clusters:
        unique = np.unique(current_labels)
        counts = {l: np.sum(current_labels == l) for l in unique}
        largest = max(counts, key=counts.get)
        largest_mask = current_labels == largest
        
        if verbose:
            print(f"      Splitting cluster {largest} ({counts[largest]} pts)")
        
        # Bisect the largest cluster
        sub_coords = coords[largest_mask]
        sub_values = values[largest_mask]
        sub_labels = spatial_bisection(sub_coords, sub_values)
        
        # Assign new labels
        current_labels[largest_mask] = np.where(sub_labels == 0, largest, next_label)
        next_label += 1
    
    # Relabel to 0, 1, ..., n_clusters-1
    unique = np.unique(current_labels)
    label_map = {old: new for new, old in enumerate(unique)}
    return np.array([label_map[l] for l in current_labels])


def spatial_bisection_n(
    coords: np.ndarray,
    values: np.ndarray,
    n_clusters: int
) -> np.ndarray:
    """
    Spatial bisection to get n_clusters using recursive splitting.
    """
    if n_clusters <= 1:
        return np.zeros(len(coords), dtype=int)
    
    # Start with one cluster
    labels = np.zeros(len(coords), dtype=int)
    
    # Recursively split until we have n_clusters
    return split_to_n_clusters(coords, values, labels, n_clusters, verbose=False)


def spatial_bisection(coords: np.ndarray, values: np.ndarray = None) -> np.ndarray:
    """
    Fallback method: bisect spatially along principal axis.
    
    Uses PCA to find the principal axis and splits at the median.
    """
    # Center coordinates
    centered = coords - coords.mean(axis=0)
    
    # PCA to find principal axis
    cov = np.cov(centered.T)
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    principal_axis = eigenvectors[:, np.argmax(eigenvalues)]
    
    # Project onto principal axis
    projections = centered @ principal_axis
    
    # Split at median
    median_proj = np.median(projections)
    labels = (projections >= median_proj).astype(int)
    
    return labels


def assign_region_labels_consistently(
    gene_labels: np.ndarray,
    gene_coords: np.ndarray,
    mz_labels: np.ndarray,
    mz_coords: np.ndarray,
    n_clusters: int = 2
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Ensure region labels are consistent between gene and m/z data.
    
    Uses centroid matching with Hungarian algorithm to align labels:
    Region i in gene should correspond spatially to Region i in m/z.
    
    Args:
        gene_labels: (N_gene,) array of gene cluster labels
        gene_coords: (N_gene, 2) array of gene coordinates (aligned to MSI)
        mz_labels: (N_mz,) array of m/z cluster labels
        mz_coords: (N_mz, 2) array of m/z coordinates
        n_clusters: Number of clusters
    
    Returns:
        Tuple of (gene_labels, mz_labels) with consistent labeling
    """
    from scipy.optimize import linear_sum_assignment
    
    # Compute centroids for each region
    gene_centroids = np.zeros((n_clusters, 2))
    mz_centroids = np.zeros((n_clusters, 2))
    
    for label in range(n_clusters):
        gene_mask = gene_labels == label
        mz_mask = mz_labels == label
        
        if gene_mask.sum() > 0:
            gene_centroids[label] = gene_coords[gene_mask].mean(axis=0)
        if mz_mask.sum() > 0:
            mz_centroids[label] = mz_coords[mz_mask].mean(axis=0)
    
    # Compute distance matrix between gene and mz centroids
    dist_matrix = cdist(gene_centroids, mz_centroids)
    
    # Use Hungarian algorithm to find optimal assignment
    gene_idx, mz_idx = linear_sum_assignment(dist_matrix)
    
    # Create mapping for mz labels
    mz_label_map = {mz_idx[i]: gene_idx[i] for i in range(n_clusters)}
    
    # Remap mz labels to match gene labels
    new_mz_labels = np.array([mz_label_map.get(l, l) for l in mz_labels])
    
    return gene_labels, new_mz_labels


# =============================================================================
# COORDINATE TRANSFORMATION
# =============================================================================

def rotate_180(coords: np.ndarray) -> np.ndarray:
    """Rotate coordinates 180° around centroid"""
    center = coords.mean(axis=0)
    return 2 * center - coords


def align_rna_to_msi(
    rna_coords: np.ndarray,
    msi_coords: np.ndarray
) -> np.ndarray:
    """
    Align RNA coordinates to MSI coordinate system.
    1. Rotate 180°
    2. Scale to match MSI extent
    """
    rotated = rotate_180(rna_coords)
    
    rna_min, rna_max = rotated.min(axis=0), rotated.max(axis=0)
    msi_min, msi_max = msi_coords.min(axis=0), msi_coords.max(axis=0)
    
    rna_range = rna_max - rna_min
    msi_range = msi_max - msi_min
    
    scale = msi_range / (rna_range + 1e-8)
    
    aligned = (rotated - rna_min) * scale + msi_min
    
    return aligned


# =============================================================================
# RESOLUTION-INVARIANT DESCRIPTORS
# =============================================================================

def compute_value_histogram(values: np.ndarray, n_bins: int = 50) -> np.ndarray:
    """Value distribution histogram"""
    hist, _ = np.histogram(values, bins=n_bins, density=True)
    return hist / (hist.sum() + 1e-8)


def compute_spatial_histogram(
    coords: np.ndarray,
    values: np.ndarray,
    n_bins: int = 10
) -> np.ndarray:
    """2D spatial histogram (normalized coordinates)"""
    norm = (coords - coords.min(axis=0)) / (coords.max(axis=0) - coords.min(axis=0) + 1e-8)
    
    hist = np.zeros((n_bins, n_bins))
    counts = np.zeros((n_bins, n_bins))
    
    for i in range(len(coords)):
        x_bin = min(int(norm[i, 0] * n_bins), n_bins - 1)
        y_bin = min(int(norm[i, 1] * n_bins), n_bins - 1)
        hist[y_bin, x_bin] += values[i]
        counts[y_bin, x_bin] += 1
    
    hist = np.divide(hist, counts, where=counts > 0, out=np.zeros_like(hist))
    
    if hist.max() > hist.min():
        hist = (hist - hist.min()) / (hist.max() - hist.min())
    
    return hist


def compute_radial_profile(
    coords: np.ndarray,
    values: np.ndarray,
    n_rings: int = 10
) -> np.ndarray:
    """Radial distribution from centroid"""
    if len(coords) == 0:
        return np.zeros(n_rings)
    
    norm = (coords - coords.min(axis=0)) / (coords.max(axis=0) - coords.min(axis=0) + 1e-8)
    centroid = norm.mean(axis=0)
    
    distances = np.linalg.norm(norm - centroid, axis=1)
    max_dist = distances.max() if distances.max() > 0 else 1
    
    profile = np.zeros(n_rings)
    counts = np.zeros(n_rings)
    
    for i in range(len(coords)):
        ring = min(int(distances[i] / max_dist * n_rings), n_rings - 1)
        profile[ring] += values[i]
        counts[ring] += 1
    
    profile = np.divide(profile, counts, where=counts > 0, out=np.zeros_like(profile))
    
    if profile.max() > profile.min():
        profile = (profile - profile.min()) / (profile.max() - profile.min())
    
    return profile


def compute_quadrant_stats(
    coords: np.ndarray,
    values: np.ndarray,
    n_div: int = 3
) -> np.ndarray:
    """Statistics in spatial quadrants"""
    if len(coords) == 0:
        return np.zeros(n_div * n_div * 2)
    
    norm = (coords - coords.min(axis=0)) / (coords.max(axis=0) - coords.min(axis=0) + 1e-8)
    
    stats = []
    for qx in range(n_div):
        for qy in range(n_div):
            x_min, x_max = qx / n_div, (qx + 1) / n_div
            y_min, y_max = qy / n_div, (qy + 1) / n_div
            
            mask = (
                (norm[:, 0] >= x_min) & (norm[:, 0] < x_max) &
                (norm[:, 1] >= y_min) & (norm[:, 1] < y_max)
            )
            
            if mask.sum() > 0:
                q_vals = values[mask]
                stats.extend([np.mean(q_vals), np.std(q_vals)])
            else:
                stats.extend([0, 0])
    
    stats = np.array(stats)
    if stats.max() > stats.min():
        stats = (stats - stats.min()) / (stats.max() - stats.min())
    
    return stats


def compute_morans_i(
    coords: np.ndarray,
    values: np.ndarray,
    k: int = 8
) -> float:
    """Moran's I spatial autocorrelation"""
    if len(coords) < k + 1:
        return 0.0
    
    norm = (coords - coords.min(axis=0)) / (coords.max(axis=0) - coords.min(axis=0) + 1e-8)
    
    nn = NearestNeighbors(n_neighbors=min(k + 1, len(coords)))
    nn.fit(norm)
    _, indices = nn.kneighbors(norm)
    
    n = len(values)
    mean_val = values.mean()
    denom = np.sum((values - mean_val) ** 2)
    
    if denom == 0:
        return 0
    
    numer = 0
    w_sum = 0
    
    for i in range(n):
        for j in indices[i, 1:]:
            numer += (values[i] - mean_val) * (values[j] - mean_val)
            w_sum += 1
    
    return (n / w_sum) * (numer / denom) if w_sum > 0 else 0


# =============================================================================
# REGION DESCRIPTOR
# =============================================================================

@dataclass
class RegionDescriptor:
    """Descriptor for a single region within a sample"""
    region_id: int
    n_points: int
    centroid: np.ndarray
    
    # Descriptors
    value_histogram: np.ndarray
    spatial_histogram: np.ndarray
    radial_profile: np.ndarray
    quadrant_stats: np.ndarray
    morans_i: float
    
    # Statistics
    mean_value: float
    std_value: float
    area_fraction: float  # Fraction of total sample points
    
    # GNN embedding (regional)
    regional_embedding: Optional[np.ndarray] = None
    mean_importance: float = 0.0


# =============================================================================
# SPATIAL SIGNATURE (ENHANCED)
# =============================================================================

@dataclass
class SpatialSignature:
    """Complete spatial signature with regional information"""
    sample_id: str
    feature_name: str
    feature_type: str
    
    # GNN embeddings
    global_embedding: np.ndarray
    node_embeddings: np.ndarray
    node_importance: np.ndarray
    
    # Global descriptors
    value_histogram: np.ndarray
    spatial_histogram: np.ndarray
    radial_profile: np.ndarray
    quadrant_stats: np.ndarray
    morans_i: float
    
    # Raw data
    coordinates: np.ndarray
    raw_values: np.ndarray
    
    # Aligned coordinates (for RNA only, after 180° rotation)
    aligned_coordinates: Optional[np.ndarray] = None
    
    # Leiden clustering results
    region_labels: Optional[np.ndarray] = None
    region_descriptors: Optional[Dict[int, RegionDescriptor]] = None


# =============================================================================
# GNN MODEL
# =============================================================================

class SpatialGNN(nn.Module):
    """Spatial GNN for embeddings"""
    
    def __init__(
        self,
        input_dim: int = None,
        hidden_dim: int = 128,
        embedding_dim: int = 64,
        num_heads: int = 4,
        num_layers: int = 3,
        projection_dim: int = 64
    ):
        super().__init__()
        
        self.projection_dim = projection_dim
        self.input_projections = nn.ModuleDict()
        
        if input_dim is not None:
            self.input_projections[str(input_dim)] = nn.Sequential(
                nn.Linear(input_dim, projection_dim),
                nn.LayerNorm(projection_dim),
                nn.GELU()
            )
        
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        
        self.convs.append(GATConv(projection_dim, hidden_dim // num_heads,
                                  heads=num_heads, concat=True, dropout=0.1))
        self.norms.append(nn.LayerNorm(hidden_dim))
        
        for _ in range(num_layers - 2):
            self.convs.append(GATConv(hidden_dim, hidden_dim // num_heads,
                                      heads=num_heads, concat=True, dropout=0.1))
            self.norms.append(nn.LayerNorm(hidden_dim))
        
        self.convs.append(GATConv(hidden_dim, embedding_dim, heads=1, concat=False, dropout=0.1))
        self.norms.append(nn.LayerNorm(embedding_dim))
        
        self.importance_head = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
        
        self.pool_attention = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, projection_dim)
        )
    
    def _get_projection(self, dim):
        key = str(dim)
        if key not in self.input_projections:
            device = next(self.parameters()).device
            self.input_projections[key] = nn.Sequential(
                nn.Linear(dim, self.projection_dim),
                nn.LayerNorm(self.projection_dim),
                nn.GELU()
            ).to(device)
        return self.input_projections[key]
    
    def forward(self, x, edge_index, bio_importance=None):
        h_input = self._get_projection(x.size(1))(x)
        
        h = h_input
        for conv, norm in zip(self.convs, self.norms):
            h = conv(h, edge_index)
            h = norm(h)
            h = F.gelu(h)
        
        node_embeddings = h
        
        learned_imp = self.importance_head(node_embeddings).squeeze(-1)
        if bio_importance is not None:
            node_importance = 0.5 * learned_imp + 0.5 * bio_importance
        else:
            node_importance = learned_imp
        node_importance = node_importance / (node_importance.max() + 1e-8)
        
        weight = node_importance / (node_importance.sum() + 1e-8) * x.size(0)
        weighted_emb = node_embeddings * weight.unsqueeze(-1)
        
        attn = F.softmax(self.pool_attention(node_embeddings).squeeze(-1), dim=0)
        global_emb = (attn.unsqueeze(-1) * weighted_emb).sum(dim=0)
        
        reconstructed = self.decoder(node_embeddings)
        
        return {
            'node_embeddings': node_embeddings,
            'global_embedding': global_emb,
            'node_importance': node_importance,
            'reconstructed': reconstructed,
            'h_input': h_input
        }


class SpatialLoss(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, output, edge_index):
        recon = F.mse_loss(output['reconstructed'], output['h_input'])
        
        src, dst = edge_index[0], edge_index[1]
        smooth = ((output['node_embeddings'][src] - output['node_embeddings'][dst]) ** 2).mean()
        
        return recon + 0.3 * smooth


# =============================================================================
# SIMILARITY COMPUTATION (GLOBAL)
# =============================================================================

def compute_coordinate_based_similarity(
    gene_sig: SpatialSignature,
    mz_sig: SpatialSignature,
    grid_res: int = 50
) -> dict:
    """Compute similarity using aligned coordinates."""
    gene_coords = gene_sig.aligned_coordinates if gene_sig.aligned_coordinates is not None \
                  else gene_sig.coordinates
    mz_coords = mz_sig.coordinates
    
    x_min = min(gene_coords[:, 0].min(), mz_coords[:, 0].min())
    x_max = max(gene_coords[:, 0].max(), mz_coords[:, 0].max())
    y_min = min(gene_coords[:, 1].min(), mz_coords[:, 1].min())
    y_max = max(gene_coords[:, 1].max(), mz_coords[:, 1].max())
    
    grid_x, grid_y = np.meshgrid(
        np.linspace(x_min, x_max, grid_res),
        np.linspace(y_min, y_max, grid_res)
    )
    
    gene_grid = griddata(gene_coords, gene_sig.raw_values, (grid_x, grid_y), method='linear')
    mz_grid = griddata(mz_coords, mz_sig.raw_values, (grid_x, grid_y), method='linear')
    
    gene_imp_grid = griddata(gene_coords, gene_sig.node_importance, (grid_x, grid_y), method='linear')
    mz_imp_grid = griddata(mz_coords, mz_sig.node_importance, (grid_x, grid_y), method='linear')
    
    mask = ~(np.isnan(gene_grid) | np.isnan(mz_grid))
    if mask.sum() > 10:
        value_corr, _ = pearsonr(gene_grid[mask], mz_grid[mask])
        value_corr = value_corr if not np.isnan(value_corr) else 0
    else:
        value_corr = 0
    
    mask_imp = ~(np.isnan(gene_imp_grid) | np.isnan(mz_imp_grid))
    if mask_imp.sum() > 0:
        g_imp = gene_imp_grid.copy()
        m_imp = mz_imp_grid.copy()
        g_imp[np.isnan(g_imp)] = 0
        m_imp[np.isnan(m_imp)] = 0
        
        g_imp = g_imp / (g_imp.max() + 1e-8)
        m_imp = m_imp / (m_imp.max() + 1e-8)
        
        intersection = np.minimum(g_imp, m_imp).sum()
        union = np.maximum(g_imp, m_imp).sum()
        importance_iou = intersection / (union + 1e-8)
        
        imp_corr, _ = pearsonr(g_imp[mask_imp], m_imp[mask_imp])
        imp_corr = imp_corr if not np.isnan(imp_corr) else 0
    else:
        importance_iou = 0
        imp_corr = 0
    
    return {
        'value_correlation': value_corr,
        'importance_iou': importance_iou,
        'importance_correlation': imp_corr
    }


def compute_descriptor_similarity(
    gene_sig: SpatialSignature,
    mz_sig: SpatialSignature
) -> dict:
    """Compute similarity using resolution-invariant descriptors"""
    
    g_emb = gene_sig.global_embedding.flatten()
    m_emb = mz_sig.global_embedding.flatten()
    emb_cosine = np.dot(g_emb, m_emb) / (np.linalg.norm(g_emb) * np.linalg.norm(m_emb) + 1e-8)
    
    if gene_sig.value_histogram.std() > 0 and mz_sig.value_histogram.std() > 0:
        val_hist_corr, _ = pearsonr(gene_sig.value_histogram, mz_sig.value_histogram)
        val_hist_corr = val_hist_corr if not np.isnan(val_hist_corr) else 0
    else:
        val_hist_corr = 0
    
    g_spatial = gene_sig.spatial_histogram.flatten()
    m_spatial = mz_sig.spatial_histogram.flatten()
    if g_spatial.std() > 0 and m_spatial.std() > 0:
        spatial_hist_corr, _ = pearsonr(g_spatial, m_spatial)
        spatial_hist_corr = spatial_hist_corr if not np.isnan(spatial_hist_corr) else 0
    else:
        spatial_hist_corr = 0
    
    if gene_sig.radial_profile.std() > 0 and mz_sig.radial_profile.std() > 0:
        radial_corr, _ = pearsonr(gene_sig.radial_profile, mz_sig.radial_profile)
        radial_corr = radial_corr if not np.isnan(radial_corr) else 0
    else:
        radial_corr = 0
    
    if gene_sig.quadrant_stats.std() > 0 and mz_sig.quadrant_stats.std() > 0:
        quad_corr, _ = pearsonr(gene_sig.quadrant_stats, mz_sig.quadrant_stats)
        quad_corr = quad_corr if not np.isnan(quad_corr) else 0
    else:
        quad_corr = 0
    
    morans_diff = abs(gene_sig.morans_i - mz_sig.morans_i)
    morans_sim = 1 / (1 + morans_diff)
    
    return {
        'embedding_cosine': emb_cosine,
        'value_hist_corr': val_hist_corr,
        'spatial_hist_corr': spatial_hist_corr,
        'radial_corr': radial_corr,
        'quadrant_corr': quad_corr,
        'morans_similarity': morans_sim
    }


# =============================================================================
# REGION-LEVEL SIMILARITY COMPUTATION
# =============================================================================

def compute_region_descriptor(
    coords: np.ndarray,
    values: np.ndarray,
    node_embeddings: np.ndarray,
    node_importance: np.ndarray,
    mask: np.ndarray,
    region_id: int,
    total_points: int
) -> RegionDescriptor:
    """
    Compute descriptor for a single region.
    
    Args:
        coords: Full coordinates array
        values: Full values array
        node_embeddings: Full node embeddings from GNN
        node_importance: Full node importance scores
        mask: Boolean mask for this region
        region_id: Region identifier
        total_points: Total points in sample (for area fraction)
    
    Returns:
        RegionDescriptor for this region
    """
    region_coords = coords[mask]
    region_values = values[mask]
    region_embeddings = node_embeddings[mask]
    region_importance = node_importance[mask]
    
    n_points = mask.sum()
    
    if n_points < 5:
        # Too few points, return empty descriptor
        return RegionDescriptor(
            region_id=region_id,
            n_points=n_points,
            centroid=region_coords.mean(axis=0) if n_points > 0 else np.zeros(2),
            value_histogram=np.zeros(50),
            spatial_histogram=np.zeros((10, 10)),
            radial_profile=np.zeros(10),
            quadrant_stats=np.zeros(18),
            morans_i=0.0,
            mean_value=region_values.mean() if n_points > 0 else 0,
            std_value=region_values.std() if n_points > 0 else 0,
            area_fraction=n_points / total_points,
            regional_embedding=region_embeddings.mean(axis=0) if n_points > 0 else None,
            mean_importance=region_importance.mean() if n_points > 0 else 0
        )
    
    return RegionDescriptor(
        region_id=region_id,
        n_points=n_points,
        centroid=region_coords.mean(axis=0),
        value_histogram=compute_value_histogram(region_values),
        spatial_histogram=compute_spatial_histogram(region_coords, region_values),
        radial_profile=compute_radial_profile(region_coords, region_values),
        quadrant_stats=compute_quadrant_stats(region_coords, region_values),
        morans_i=compute_morans_i(region_coords, region_values, k=min(8, n_points-1)),
        mean_value=region_values.mean(),
        std_value=region_values.std(),
        area_fraction=n_points / total_points,
        regional_embedding=region_embeddings.mean(axis=0),
        mean_importance=region_importance.mean()
    )


def compute_region_similarity(
    gene_region: RegionDescriptor,
    mz_region: RegionDescriptor
) -> dict:
    """
    Compute similarity between two regions.
    
    Args:
        gene_region: Gene region descriptor
        mz_region: m/z region descriptor
    
    Returns:
        Dictionary of similarity metrics
    """
    # Skip if either region is too small
    if gene_region.n_points < 5 or mz_region.n_points < 5:
        return {
            'embedding_cosine': 0,
            'value_hist_corr': 0,
            'spatial_hist_corr': 0,
            'radial_corr': 0,
            'morans_similarity': 0,
            'importance_similarity': 0,
            'region_score': 0
        }
    
    # Embedding cosine
    if gene_region.regional_embedding is not None and mz_region.regional_embedding is not None:
        g_emb = gene_region.regional_embedding.flatten()
        m_emb = mz_region.regional_embedding.flatten()
        emb_cosine = np.dot(g_emb, m_emb) / (np.linalg.norm(g_emb) * np.linalg.norm(m_emb) + 1e-8)
    else:
        emb_cosine = 0
    
    # Value histogram correlation
    if gene_region.value_histogram.std() > 0 and mz_region.value_histogram.std() > 0:
        val_hist_corr, _ = pearsonr(gene_region.value_histogram, mz_region.value_histogram)
        val_hist_corr = val_hist_corr if not np.isnan(val_hist_corr) else 0
    else:
        val_hist_corr = 0
    
    # Spatial histogram correlation
    g_spatial = gene_region.spatial_histogram.flatten()
    m_spatial = mz_region.spatial_histogram.flatten()
    if g_spatial.std() > 0 and m_spatial.std() > 0:
        spatial_hist_corr, _ = pearsonr(g_spatial, m_spatial)
        spatial_hist_corr = spatial_hist_corr if not np.isnan(spatial_hist_corr) else 0
    else:
        spatial_hist_corr = 0
    
    # Radial profile correlation
    if gene_region.radial_profile.std() > 0 and mz_region.radial_profile.std() > 0:
        radial_corr, _ = pearsonr(gene_region.radial_profile, mz_region.radial_profile)
        radial_corr = radial_corr if not np.isnan(radial_corr) else 0
    else:
        radial_corr = 0
    
    # Moran's I similarity
    morans_diff = abs(gene_region.morans_i - mz_region.morans_i)
    morans_sim = 1 / (1 + morans_diff)
    
    # Importance similarity
    imp_diff = abs(gene_region.mean_importance - mz_region.mean_importance)
    imp_sim = 1 / (1 + imp_diff)
    
    # Combined region score
    region_score = (
        0.25 * max(emb_cosine, 0) +
        0.20 * max(val_hist_corr, 0) +
        0.20 * max(spatial_hist_corr, 0) +
        0.15 * max(radial_corr, 0) +
        0.10 * morans_sim +
        0.10 * imp_sim
    )
    
    return {
        'embedding_cosine': emb_cosine,
        'value_hist_corr': val_hist_corr,
        'spatial_hist_corr': spatial_hist_corr,
        'radial_corr': radial_corr,
        'morans_similarity': morans_sim,
        'importance_similarity': imp_sim,
        'region_score': region_score
    }


def compute_regional_similarity(
    gene_sig: SpatialSignature,
    mz_sig: SpatialSignature
) -> dict:
    """
    Compute region-to-region similarity.
    
    Compares Region i (gene) to Region i (m/z) for all regions.
    
    Args:
        gene_sig: Gene signature with region info
        mz_sig: m/z signature with region info
    
    Returns:
        Dictionary with regional similarity metrics
    """
    if gene_sig.region_descriptors is None or mz_sig.region_descriptors is None:
        return {
            'region_scores': {},
            'avg_region_score': 0,
            'weighted_region_score': 0
        }
    
    # Get number of regions
    n_regions = len(gene_sig.region_descriptors)
    
    # Compare matching regions
    region_scores = {}
    region_details = {}
    total_weight = 0
    weighted_sum = 0
    
    for region_id in range(n_regions):
        if region_id in gene_sig.region_descriptors and region_id in mz_sig.region_descriptors:
            region_sim = compute_region_similarity(
                gene_sig.region_descriptors[region_id],
                mz_sig.region_descriptors[region_id]
            )
            region_scores[region_id] = region_sim['region_score']
            region_details[region_id] = region_sim
            
            # Weight by gene region area
            weight = gene_sig.region_descriptors[region_id].area_fraction
            weighted_sum += weight * region_sim['region_score']
            total_weight += weight
    
    avg_score = np.mean(list(region_scores.values())) if region_scores else 0
    weighted_score = weighted_sum / total_weight if total_weight > 0 else 0
    
    result = {
        'region_scores': region_scores,
        'region_details': region_details,
        'avg_region_score': avg_score,
        'weighted_region_score': weighted_score
    }
    
    # Add individual region scores for backward compatibility
    for region_id, score in region_scores.items():
        result[f'region_{region_id}_score'] = score
    
    return result


# =============================================================================
# COMBINED SCORING
# =============================================================================

def compute_combined_score(
    coord_sim: dict,
    desc_sim: dict,
    regional_sim: Optional[dict] = None
) -> Tuple[float, dict]:
    """
    Combined score using global, coordinate-based, descriptor-based,
    and regional metrics.
    
    Weights:
    - Coordinate-based (30%): Direct spatial comparison after alignment
    - Descriptor-based (30%): Resolution-invariant for robustness
    - Regional (40%): Region-to-region matching
    
    Returns:
        Tuple of (combined_score, score_breakdown)
    """
    # Coordinate-based (30%)
    coord_score = (
        0.12 * max(coord_sim['value_correlation'], 0) +
        0.09 * coord_sim['importance_iou'] +
        0.09 * max(coord_sim['importance_correlation'], 0)
    )
    
    # Descriptor-based (30%)
    desc_score = (
        0.10 * desc_sim['embedding_cosine'] +
        0.06 * max(desc_sim['spatial_hist_corr'], 0) +
        0.06 * max(desc_sim['radial_corr'], 0) +
        0.03 * max(desc_sim['quadrant_corr'], 0) +
        0.03 * desc_sim['morans_similarity'] +
        0.02 * max(desc_sim['value_hist_corr'], 0)
    )
    
    # Regional (40%)
    if regional_sim is not None:
        regional_score = 0.40 * regional_sim['weighted_region_score']
    else:
        # If no regional data, redistribute to coord and desc
        regional_score = 0
        coord_score *= (0.30 + 0.20) / 0.30  # Boost
        desc_score *= (0.30 + 0.20) / 0.30
    
    combined = coord_score + desc_score + regional_score
    
    breakdown = {
        'coordinate_component': coord_score,
        'descriptor_component': desc_score,
        'regional_component': regional_score,
        'combined': combined
    }
    
    return combined, breakdown


# =============================================================================
# MAIN MATCHER
# =============================================================================

class HybridMatcherWithKMeans:
    """Hybrid matcher with pre-computed clustering for regional analysis"""
    
    def __init__(
        self,
        output_dir: str = './gene_to_mz_results_v8_kmeans',
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    ):
        self.device = device
        self.output_dir = output_dir
        
        subdirs = ['saliency_maps', 'gene_visualizations', 'match_visualizations',
                   'embeddings', 'training_curves', 'descriptors', 'region_maps']
        for subdir in subdirs:
            os.makedirs(os.path.join(output_dir, subdir), exist_ok=True)
        
        self.rna_data = {}
        self.msi_data = {}
        self.rna_model = None
        self.msi_model = None
        
        # Pre-computed region labels (computed once per sample)
        self.rna_region_labels = {}  # {sample_id: np.ndarray}
        self.msi_region_labels = {}  # {sample_id: np.ndarray}
        self.rna_aligned_coords = {}  # {sample_id: np.ndarray} - aligned RNA coords
    
    def load_all_data(self):
        """Load data"""
        print("Loading RNA-seq data...")
        print(f"  Pixel size: {RNA_PIXEL_SIZE} μm (hexagonal)")
        for file, sample_id in zip(RNA_SAMPLE_FILES, RNA_SAMPLE_IDS):
            path = os.path.join(RNA_INPUT_FOLDER, file)
            if os.path.exists(path):
                self.rna_data[sample_id] = sc.read_h5ad(path)
                print(f"  {sample_id}: {self.rna_data[sample_id].shape}")
        
        print(f"\nLoading MSI data...")
        print(f"  Pixel size: {MSI_PIXEL_SIZE} μm (Cartesian)")
        for file, sample_id in zip(MSI_SAMPLE_FILES, MSI_SAMPLE_IDS):
            path = os.path.join(MSI_INPUT_FOLDER, file)
            if os.path.exists(path):
                self.msi_data[sample_id] = sc.read_h5ad(path)
                print(f"  {sample_id}: {self.msi_data[sample_id].shape}")
    
    def precompute_region_labels(self, method: str = 'kmeans', n_clusters: int = 2):
        """
        Pre-compute region labels for all samples using clustering.
        
        This is called ONCE at the beginning before any matching.
        Region labels are based on INTENSITY VALUES (scanpy-style pipeline):
        1. StandardScaler on intensity matrix
        2. PCA reduction
        3. KMeans or Leiden clustering
        
        Args:
            method: 'kmeans' or 'leiden'
            n_clusters: Number of regions to create
        """
        print("\n" + "="*70)
        print(f"PRE-COMPUTING REGION LABELS ({method.upper()}, {n_clusters} clusters)")
        print("Using intensity-based clustering (StandardScaler -> PCA -> Clustering)")
        print("="*70)
        
        # First, compute MSI region labels (these are the reference)
        print("\nClustering MSI samples (based on m/z intensities)...")
        for sample_id, adata in self.msi_data.items():
            coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
            
            # Get intensity matrix (cells × features)
            X = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
            
            print(f"  {sample_id}: {X.shape[0]} cells × {X.shape[1]} features")
            
            labels = cluster_regions(
                X=X,
                coords=coords,
                n_clusters=n_clusters,
                method=method,
                n_neighbors=15,  # Standard for KNN graph
                n_pca_components=30,
                seed=42,
                verbose=True
            )
            
            self.msi_region_labels[sample_id] = labels
            
            # Print cluster info
            unique, counts = np.unique(labels, return_counts=True)
            size_str = ", ".join([f"R{u}={c}" for u, c in zip(unique, counts)])
            print(f"    Clusters: {size_str}")
        
        # Then compute RNA region labels with alignment to MSI
        print("\nClustering RNA samples (based on gene expression)...")
        for sample_id, adata in self.rna_data.items():
            rna_coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
            
            # Get intensity matrix (cells × features)
            X = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
            
            print(f"  {sample_id}: {X.shape[0]} cells × {X.shape[1]} features")
            
            # Check if matching MSI sample exists for alignment
            if sample_id in self.msi_data:
                msi_adata = self.msi_data[sample_id]
                msi_coords = np.column_stack([msi_adata.obs['x_um'].values, msi_adata.obs['y_um'].values])
                
                # Align RNA to MSI (180° rotation + scaling)
                aligned_coords = align_rna_to_msi(rna_coords, msi_coords)
                self.rna_aligned_coords[sample_id] = aligned_coords
                
                # Cluster using intensity matrix
                labels = cluster_regions(
                    X=X,
                    coords=aligned_coords,
                    n_clusters=n_clusters,
                    method=method,
                    n_neighbors=15,
                    n_pca_components=30,
                    seed=42,
                    verbose=True
                )
                
                # Align labels to match MSI regions
                # We want to remap RNA labels to match MSI region spatial positions
                if sample_id in self.msi_region_labels:
                    # The function remaps the second argument's labels to match the first
                    # We want RNA labels remapped to match MSI, so swap the order
                    _, labels = assign_region_labels_consistently(
                        self.msi_region_labels[sample_id], msi_coords,
                        labels, aligned_coords,
                        n_clusters=n_clusters
                    )
            else:
                # No matching MSI, just cluster RNA
                aligned_coords = rna_coords
                self.rna_aligned_coords[sample_id] = aligned_coords
                
                labels = cluster_regions(
                    X=X,
                    coords=rna_coords,
                    n_clusters=n_clusters,
                    method=method,
                    n_neighbors=15,
                    n_pca_components=30,
                    seed=42,
                    verbose=True
                )
            
            self.rna_region_labels[sample_id] = labels
            
            # Print cluster info
            unique, counts = np.unique(labels, return_counts=True)
            size_str = ", ".join([f"R{u}={c}" for u, c in zip(unique, counts)])
            print(f"    Clusters: {size_str}")
        
        # Save region visualizations
        print("\nSaving region maps...")
        self._visualize_region_maps(n_clusters)
        
        print(f"\nRegion labels pre-computed for {len(self.rna_region_labels)} RNA and {len(self.msi_region_labels)} MSI samples")
    
    def _visualize_region_maps(self, n_clusters: int):
        """Save visualizations of the pre-computed region labels"""
        cmap = plt.cm.get_cmap('tab10', n_clusters)
        colors = [cmap(i) for i in range(n_clusters)]
        
        # Visualize RNA regions
        for sample_id in self.rna_region_labels:
            if sample_id not in self.rna_data:
                continue
            
            labels = self.rna_region_labels[sample_id]
            
            # Use aligned coords if available, otherwise get from adata
            if sample_id in self.rna_aligned_coords:
                coords = self.rna_aligned_coords[sample_id]
            else:
                adata = self.rna_data[sample_id]
                coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
            
            # Verify dimensions match
            if len(coords) != len(labels):
                print(f"  Warning: Skipping RNA {sample_id} visualization - dimension mismatch (coords={len(coords)}, labels={len(labels)})")
                continue
            
            fig, ax = plt.subplots(1, 1, figsize=(10, 10))
            for region_id in range(n_clusters):
                mask = labels == region_id
                ax.scatter(coords[mask, 0], coords[mask, 1],
                          c=[colors[region_id]], s=15, alpha=0.7,
                          label=f'Region {region_id} ({mask.sum()} pts)')
            ax.set_title(f'RNA {sample_id} - Pre-computed Regions', fontweight='bold')
            ax.legend()
            ax.set_aspect('equal')
            plt.savefig(os.path.join(self.output_dir, 'region_maps', f'rna_{sample_id}_regions.png'),
                       dpi=150, bbox_inches='tight')
            plt.close()
        
        # Visualize MSI regions
        for sample_id in self.msi_region_labels:
            if sample_id not in self.msi_data:
                continue
            
            adata = self.msi_data[sample_id]
            coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
            labels = self.msi_region_labels[sample_id]
            
            # Verify dimensions match
            if len(coords) != len(labels):
                print(f"  Warning: Skipping MSI {sample_id} visualization - dimension mismatch (coords={len(coords)}, labels={len(labels)})")
                continue
            
            fig, ax = plt.subplots(1, 1, figsize=(10, 10))
            for region_id in range(n_clusters):
                mask = labels == region_id
                ax.scatter(coords[mask, 0], coords[mask, 1],
                          c=[colors[region_id]], s=5, alpha=0.7,
                          label=f'Region {region_id} ({mask.sum()} pts)')
            ax.set_title(f'MSI {sample_id} - Pre-computed Regions', fontweight='bold')
            ax.legend()
            ax.set_aspect('equal')
            plt.savefig(os.path.join(self.output_dir, 'region_maps', f'msi_{sample_id}_regions.png'),
                       dpi=150, bbox_inches='tight')
            plt.close()
    
    def build_graph(self, coords, n_neighbors):
        """Build spatial graph"""
        nn = NearestNeighbors(n_neighbors=n_neighbors + 1)
        nn.fit(coords)
        distances, indices = nn.kneighbors(coords)
        
        median_dist = np.median(distances[:, 1])
        max_dist = median_dist * 1.5
        
        edges = set()
        for i in range(len(coords)):
            for j_idx in range(1, n_neighbors + 1):
                if distances[i, j_idx] <= max_dist:
                    j = indices[i, j_idx]
                    edges.add((i, j))
                    edges.add((j, i))
        
        return torch.tensor(list(edges), dtype=torch.long).t().contiguous()
    
    def compute_bio_importance(self, coords, values, k):
        """Biological importance"""
        nn = NearestNeighbors(n_neighbors=k + 1)
        nn.fit(coords)
        _, indices = nn.kneighbors(coords)
        
        local_var = np.array([np.var(values[indices[i, 1:]]) for i in range(len(coords))])
        local_var = (local_var - local_var.min()) / (local_var.max() - local_var.min() + 1e-8)
        
        val_norm = (values - values.min()) / (values.max() - values.min() + 1e-8)
        
        return 0.5 * local_var + 0.5 * val_norm
    
    def prepare_data(self, coords, features, n_neighbors):
        """Prepare graph data"""
        if features.ndim == 1:
            features = features.reshape(-1, 1)
        
        bio_imp = np.zeros(len(coords))
        for col in range(features.shape[1]):
            bio_imp += self.compute_bio_importance(coords, features[:, col], n_neighbors)
        bio_imp /= features.shape[1]
        
        scaler = RobustScaler()
        features_scaled = scaler.fit_transform(features)
        
        edge_index = self.build_graph(coords, n_neighbors)
        
        return Data(x=torch.tensor(features_scaled, dtype=torch.float32), edge_index=edge_index), bio_imp
    
    def train_model(self, data_dict, bio_dict, model_type, epochs=150):
        """Train model"""
        print(f"\nTraining {model_type.upper()} model...")
        
        first_data = list(data_dict.values())[0]
        model = SpatialGNN(input_dim=first_data.x.size(1)).to(self.device)
        
        optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
        loss_fn = SpatialLoss()
        
        losses = []
        model.train()
        
        for epoch in range(epochs):
            total_loss = 0
            for sample_id, data in data_dict.items():
                data = data.to(self.device)
                bio_imp = torch.tensor(bio_dict[sample_id], dtype=torch.float32, device=self.device)
                
                optimizer.zero_grad()
                output = model(data.x, data.edge_index, bio_imp)
                loss = loss_fn(output, data.edge_index)
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
            
            losses.append(total_loss / len(data_dict))
            if (epoch + 1) % 30 == 0:
                print(f"  Epoch {epoch+1}/{epochs}, Loss: {losses[-1]:.4f}")
        
        print(f"  Final loss: {losses[-1]:.4f}")
        
        plt.figure(figsize=(10, 5))
        plt.plot(losses)
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title(f'{model_type.upper()} Training')
        plt.savefig(os.path.join(self.output_dir, 'training_curves', f'{model_type}_loss.png'), dpi=150)
        plt.close()
        
        return model
    
    def extract_signature(
        self,
        coords: np.ndarray,
        values: np.ndarray,
        sample_id: str,
        feature_name: str,
        feature_type: str,
        model: SpatialGNN,
        n_neighbors: int,
        msi_coords: Optional[np.ndarray] = None,
        region_labels: Optional[np.ndarray] = None,
        verbose: bool = False
    ) -> SpatialSignature:
        """
        Extract signature with all descriptors using pre-computed region labels.
        
        Args:
            coords: Spatial coordinates
            values: Feature values
            sample_id: Sample identifier
            feature_name: Name of the feature (gene or m/z)
            feature_type: 'gene' or 'mz'
            model: Trained GNN model
            n_neighbors: Number of neighbors for graph
            msi_coords: MSI coordinates for alignment (for genes only)
            region_labels: Pre-computed region labels (from precompute_region_labels)
            verbose: Print debug info
        """
        
        # GNN embeddings
        bio_imp = self.compute_bio_importance(coords, values, n_neighbors)
        data, _ = self.prepare_data(coords, values, n_neighbors)
        data = data.to(self.device)
        bio_tensor = torch.tensor(bio_imp, dtype=torch.float32, device=self.device)
        
        model.eval()
        with torch.no_grad():
            output = model(data.x, data.edge_index, bio_tensor)
        
        node_embeddings = output['node_embeddings'].cpu().numpy()
        node_importance = output['node_importance'].cpu().numpy()
        
        # Global descriptors
        val_hist = compute_value_histogram(values)
        spatial_hist = compute_spatial_histogram(coords, values)
        radial = compute_radial_profile(coords, values)
        quadrant = compute_quadrant_stats(coords, values)
        morans = compute_morans_i(coords, values, n_neighbors)
        
        # Aligned coordinates for RNA (use pre-computed if available)
        aligned = None
        if feature_type == 'gene':
            if sample_id in self.rna_aligned_coords:
                aligned = self.rna_aligned_coords[sample_id]
            elif msi_coords is not None:
                aligned = align_rna_to_msi(coords, msi_coords)
        
        # Use pre-computed region labels
        region_descriptors = None
        
        if region_labels is not None:
            # Use the display coordinates for region descriptors
            display_coords = aligned if aligned is not None else coords
            
            # Compute region descriptors using pre-computed labels
            region_descriptors = {}
            n_clusters = len(np.unique(region_labels))
            for region_id in range(n_clusters):
                mask = region_labels == region_id
                region_descriptors[region_id] = compute_region_descriptor(
                    display_coords, values, node_embeddings, node_importance,
                    mask, region_id, len(coords)
                )
        
        return SpatialSignature(
            sample_id=sample_id,
            feature_name=feature_name,
            feature_type=feature_type,
            global_embedding=output['global_embedding'].cpu().numpy(),
            node_embeddings=node_embeddings,
            node_importance=node_importance,
            value_histogram=val_hist,
            spatial_histogram=spatial_hist,
            radial_profile=radial,
            quadrant_stats=quadrant,
            morans_i=morans,
            coordinates=coords,
            raw_values=values,
            aligned_coordinates=aligned,
            region_labels=region_labels,
            region_descriptors=region_descriptors
        )
    
    def align_region_labels(
        self,
        gene_sig: SpatialSignature,
        mz_sig: SpatialSignature
    ) -> Tuple[SpatialSignature, SpatialSignature]:
        """
        Align region labels between gene and m/z signatures.
        Modifies signatures in place to ensure consistent labeling.
        """
        if gene_sig.region_labels is None or mz_sig.region_labels is None:
            return gene_sig, mz_sig
        
        gene_coords = gene_sig.aligned_coordinates if gene_sig.aligned_coordinates is not None \
                      else gene_sig.coordinates
        
        gene_labels, new_mz_labels = assign_region_labels_consistently(
            gene_sig.region_labels, gene_coords,
            mz_sig.region_labels, mz_sig.coordinates,
            n_clusters=N_CLUSTERS
        )
        
        # Update mz labels if they changed
        if not np.array_equal(new_mz_labels, mz_sig.region_labels):
            # Create new label mapping
            old_to_new = {}
            for old_label, new_label in zip(mz_sig.region_labels, new_mz_labels):
                if old_label not in old_to_new:
                    old_to_new[old_label] = new_label
            
            mz_sig.region_labels = new_mz_labels
            
            # Remap region descriptors
            if mz_sig.region_descriptors is not None:
                new_descriptors = {}
                for old_label, new_label in old_to_new.items():
                    if old_label in mz_sig.region_descriptors:
                        desc = mz_sig.region_descriptors[old_label]
                        desc.region_id = new_label
                        new_descriptors[new_label] = desc
                mz_sig.region_descriptors = new_descriptors
        
        return gene_sig, mz_sig
    
    def visualize_signature_with_regions(self, sig: SpatialSignature, save_path: str):
        """Visualize signature with region clustering"""
        n_regions = len(sig.region_descriptors) if sig.region_descriptors else N_CLUSTERS
        
        # Dynamic figure size based on number of regions
        n_cols = 4
        n_rows = 2 + (n_regions + 1) // 2  # Extra rows for region details
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
        
        # Row 0
        im1 = axes[0, 0].scatter(sig.coordinates[:, 0], sig.coordinates[:, 1],
                                  c=sig.raw_values, cmap='viridis', s=10)
        axes[0, 0].set_title(f'{sig.feature_name} (Original)', fontweight='bold')
        plt.colorbar(im1, ax=axes[0, 0])
        
        im2 = axes[0, 1].scatter(sig.coordinates[:, 0], sig.coordinates[:, 1],
                                  c=sig.node_importance, cmap='hot', s=10)
        axes[0, 1].set_title('Importance', fontweight='bold')
        plt.colorbar(im2, ax=axes[0, 1])
        
        # Region clustering
        if sig.region_labels is not None:
            display_coords = sig.aligned_coordinates if sig.aligned_coordinates is not None \
                            else sig.coordinates
            cmap = plt.cm.get_cmap('tab10', n_regions)
            colors = [cmap(i) for i in range(n_regions)]
            
            for region_id in range(n_regions):
                mask = sig.region_labels == region_id
                axes[0, 2].scatter(display_coords[mask, 0], display_coords[mask, 1],
                                  c=[colors[region_id]], s=10, alpha=0.7,
                                  label=f'R{region_id} ({mask.sum()})')
            axes[0, 2].set_title(f'K-means Clusters ({n_regions} regions)', fontweight='bold')
            axes[0, 2].legend(loc='best', fontsize=8)
        else:
            axes[0, 2].axis('off')
        
        im4 = axes[0, 3].imshow(sig.spatial_histogram, cmap='viridis', origin='lower')
        axes[0, 3].set_title('Spatial Histogram', fontweight='bold')
        plt.colorbar(im4, ax=axes[0, 3])
        
        # Row 1
        axes[1, 0].bar(range(len(sig.value_histogram)), sig.value_histogram)
        axes[1, 0].set_title('Value Distribution', fontweight='bold')
        
        axes[1, 1].plot(sig.radial_profile, 'b-', linewidth=2)
        axes[1, 1].set_title('Radial Profile', fontweight='bold')
        
        # Region-specific visualizations
        if sig.region_labels is not None and sig.region_descriptors is not None:
            display_coords = sig.aligned_coordinates if sig.aligned_coordinates is not None \
                            else sig.coordinates
            
            # Plot each region
            plot_idx = 0
            for region_id in range(n_regions):
                row = 1 + (plot_idx + 2) // n_cols
                col = (plot_idx + 2) % n_cols
                
                if row < n_rows:
                    mask = sig.region_labels == region_id
                    rd = sig.region_descriptors.get(region_id)
                    
                    axes[row, col].scatter(display_coords[~mask, 0], display_coords[~mask, 1],
                                           c='lightgray', s=3, alpha=0.3)
                    if mask.sum() > 0:
                        im = axes[row, col].scatter(display_coords[mask, 0], display_coords[mask, 1],
                                               c=sig.raw_values[mask], cmap='viridis', s=10)
                    
                    if rd:
                        axes[row, col].set_title(f'Region {region_id}: {mask.sum()} pts\n'
                                                f'μ={rd.mean_value:.2f}, Moran={rd.morans_i:.2f}',
                                                fontweight='bold')
                    else:
                        axes[row, col].set_title(f'Region {region_id}: {mask.sum()} pts', fontweight='bold')
                    
                    plot_idx += 1
        
        # Turn off unused axes
        for row in range(n_rows):
            for col in range(n_cols):
                if row > 1 or (row == 1 and col > 1):
                    if not axes[row, col].has_data():
                        axes[row, col].axis('off')
        
        plt.suptitle(f'{sig.feature_type.upper()}: {sig.feature_name} ({sig.sample_id}) | '
                    f'Moran\'s I: {sig.morans_i:.3f}',
                    fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        plt.close()
    
    def visualize_match_with_regions(
        self,
        gene_sig: SpatialSignature,
        mz_sig: SpatialSignature,
        coord_sim: dict,
        desc_sim: dict,
        regional_sim: dict,
        combined_score: float,
        save_path: str
    ):
        """Visualize match with regional comparison"""
        n_regions = len(gene_sig.region_descriptors) if gene_sig.region_descriptors else N_CLUSTERS
        
        # Dynamic figure layout
        n_rows = 4 + (n_regions + 1) // 2  # Extra rows for region comparisons
        fig, axes = plt.subplots(n_rows, 4, figsize=(22, 5.5 * n_rows))
        
        cmap = plt.cm.get_cmap('tab10', n_regions)
        colors = [cmap(i) for i in range(n_regions)]
        
        # Row 0: Gene overview
        im1 = axes[0, 0].scatter(gene_sig.coordinates[:, 0], gene_sig.coordinates[:, 1],
                                  c=gene_sig.raw_values, cmap='viridis', s=15)
        axes[0, 0].set_title(f'Gene: {gene_sig.feature_name} (Original)', fontweight='bold')
        plt.colorbar(im1, ax=axes[0, 0])
        
        if gene_sig.aligned_coordinates is not None:
            im2 = axes[0, 1].scatter(gene_sig.aligned_coordinates[:, 0],
                                      gene_sig.aligned_coordinates[:, 1],
                                      c=gene_sig.raw_values, cmap='viridis', s=15)
            axes[0, 1].set_title('Gene (180° Aligned)', fontweight='bold')
            plt.colorbar(im2, ax=axes[0, 1])
        
        # Gene regions
        if gene_sig.region_labels is not None:
            display_coords = gene_sig.aligned_coordinates if gene_sig.aligned_coordinates is not None \
                            else gene_sig.coordinates
            for region_id in range(n_regions):
                mask = gene_sig.region_labels == region_id
                axes[0, 2].scatter(display_coords[mask, 0], display_coords[mask, 1],
                                  c=[colors[region_id]], s=15, alpha=0.7,
                                  label=f'R{region_id} ({mask.sum()})')
            axes[0, 2].set_title('Gene Regions', fontweight='bold')
            axes[0, 2].legend(fontsize=8)
        
        im4 = axes[0, 3].imshow(gene_sig.spatial_histogram, cmap='viridis', origin='lower')
        axes[0, 3].set_title('Gene Spatial Hist', fontweight='bold')
        plt.colorbar(im4, ax=axes[0, 3])
        
        # Row 1: m/z overview
        im5 = axes[1, 0].scatter(mz_sig.coordinates[:, 0], mz_sig.coordinates[:, 1],
                                  c=mz_sig.raw_values, cmap='viridis', s=5)
        axes[1, 0].set_title(f'm/z: {mz_sig.feature_name}', fontweight='bold')
        plt.colorbar(im5, ax=axes[1, 0])
        
        im6 = axes[1, 1].scatter(mz_sig.coordinates[:, 0], mz_sig.coordinates[:, 1],
                                  c=mz_sig.node_importance, cmap='hot', s=5)
        axes[1, 1].set_title('m/z Importance', fontweight='bold')
        plt.colorbar(im6, ax=axes[1, 1])
        
        # m/z regions
        if mz_sig.region_labels is not None:
            for region_id in range(n_regions):
                mask = mz_sig.region_labels == region_id
                axes[1, 2].scatter(mz_sig.coordinates[mask, 0], mz_sig.coordinates[mask, 1],
                                  c=[colors[region_id]], s=5, alpha=0.7,
                                  label=f'R{region_id} ({mask.sum()})')
            axes[1, 2].set_title('m/z Regions', fontweight='bold')
            axes[1, 2].legend(fontsize=8)
        
        im8 = axes[1, 3].imshow(mz_sig.spatial_histogram, cmap='viridis', origin='lower')
        axes[1, 3].set_title('m/z Spatial Hist', fontweight='bold')
        plt.colorbar(im8, ax=axes[1, 3])
        
        # Row 2+: Region comparisons (2 regions per row)
        if gene_sig.region_labels is not None and mz_sig.region_labels is not None:
            gene_display = gene_sig.aligned_coordinates if gene_sig.aligned_coordinates is not None \
                          else gene_sig.coordinates
            
            for region_id in range(n_regions):
                row = 2 + region_id // 2
                col_offset = (region_id % 2) * 2
                
                gene_mask = gene_sig.region_labels == region_id
                mz_mask = mz_sig.region_labels == region_id
                
                # Gene region values
                axes[row, col_offset].scatter(gene_display[~gene_mask, 0], gene_display[~gene_mask, 1],
                                             c='lightgray', s=3, alpha=0.2)
                if gene_mask.sum() > 0:
                    axes[row, col_offset].scatter(gene_display[gene_mask, 0], gene_display[gene_mask, 1],
                                                 c=gene_sig.raw_values[gene_mask], cmap='viridis', s=15)
                axes[row, col_offset].set_title(f'Gene Region {region_id}', fontweight='bold')
                
                # m/z region values
                axes[row, col_offset+1].scatter(mz_sig.coordinates[~mz_mask, 0], mz_sig.coordinates[~mz_mask, 1],
                                               c='lightgray', s=2, alpha=0.2)
                if mz_mask.sum() > 0:
                    axes[row, col_offset+1].scatter(mz_sig.coordinates[mz_mask, 0], mz_sig.coordinates[mz_mask, 1],
                                                   c=mz_sig.raw_values[mz_mask], cmap='viridis', s=5)
                axes[row, col_offset+1].set_title(f'm/z Region {region_id}', fontweight='bold')
        
        # Last row: Metrics
        last_row = n_rows - 1
        
        # Radial profiles
        axes[last_row, 0].plot(gene_sig.radial_profile, 'b-', linewidth=2, label='Gene')
        axes[last_row, 0].plot(mz_sig.radial_profile, 'r--', linewidth=2, label='m/z')
        axes[last_row, 0].set_title(f'Radial (r={desc_sim["radial_corr"]:.3f})', fontweight='bold')
        axes[last_row, 0].legend()
        
        # Value histograms
        axes[last_row, 1].bar(range(len(gene_sig.value_histogram)), gene_sig.value_histogram, alpha=0.5, label='Gene')
        axes[last_row, 1].bar(range(len(mz_sig.value_histogram)), mz_sig.value_histogram, alpha=0.5, label='m/z')
        axes[last_row, 1].set_title(f'Value Hist (r={desc_sim["value_hist_corr"]:.3f})', fontweight='bold')
        axes[last_row, 1].legend()
        
        # Region scores bar chart
        region_scores = regional_sim.get('region_scores', {})
        if region_scores:
            x_labels = [f'R{i}' for i in range(n_regions)]
            scores = [region_scores.get(i, 0) for i in range(n_regions)]
            bar_colors = [colors[i] for i in range(n_regions)]
            bars = axes[last_row, 2].bar(x_labels, scores, color=bar_colors)
            axes[last_row, 2].set_ylim(0, 1)
            axes[last_row, 2].set_title(f'Region Scores\nAvg: {regional_sim["avg_region_score"]:.3f}', fontweight='bold')
            for bar, score in zip(bars, scores):
                axes[last_row, 2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                                      f'{score:.2f}', ha='center', fontsize=9)
        
        # Metrics summary
        region_score_text = "\n".join([f"  Region {i} score:       {region_scores.get(i, 0):.4f}" 
                                       for i in range(n_regions)])
        
        metrics_text = f"""
COMBINED SCORE: {combined_score:.4f}
═══════════════════════════════════

COORDINATE-BASED (30%):
  Value correlation:    {coord_sim['value_correlation']:.4f}
  Importance IoU:       {coord_sim['importance_iou']:.4f}
  Importance corr:      {coord_sim['importance_correlation']:.4f}

DESCRIPTOR-BASED (30%):
  Embedding cosine:     {desc_sim['embedding_cosine']:.4f}
  Spatial hist corr:    {desc_sim['spatial_hist_corr']:.4f}
  Radial corr:          {desc_sim['radial_corr']:.4f}

REGIONAL (40%):
{region_score_text}
  Weighted avg:         {regional_sim.get('weighted_region_score', 0):.4f}

SPATIAL STATS:
  Gene Moran's I:       {gene_sig.morans_i:.4f}
  m/z Moran's I:        {mz_sig.morans_i:.4f}
        """
        axes[last_row, 3].text(0.05, 0.95, metrics_text, transform=axes[last_row, 3].transAxes,
                              fontsize=9, verticalalignment='top', family='monospace',
                              bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        axes[last_row, 3].axis('off')
        
        # Turn off unused axes
        for row in range(n_rows):
            for col in range(4):
                if not axes[row, col].has_data() and row != last_row:
                    axes[row, col].axis('off')
        
        plt.suptitle(f'Match: {gene_sig.feature_name} ↔ m/z {mz_sig.feature_name} | '
                    f'Combined Score: {combined_score:.3f}',
                    fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        plt.close()
    
    def find_matches(self, gene_sig, mz_sigs, top_k=20):
        """Find matches with regional similarity"""
        matches = []
        
        for mz_name, mz_sig in mz_sigs.items():
            # Align region labels
            gene_aligned, mz_aligned = self.align_region_labels(gene_sig, mz_sig)
            
            coord_sim = compute_coordinate_based_similarity(gene_aligned, mz_aligned)
            desc_sim = compute_descriptor_similarity(gene_aligned, mz_aligned)
            regional_sim = compute_regional_similarity(gene_aligned, mz_aligned)
            
            combined, breakdown = compute_combined_score(coord_sim, desc_sim, regional_sim)
            
            match_dict = {
                'gene': gene_sig.feature_name,
                'rna_sample': gene_sig.sample_id,
                'mz_feature': mz_name,
                'msi_sample': mz_sig.sample_id,
                # Coordinate-based
                'value_correlation': coord_sim['value_correlation'],
                'importance_iou': coord_sim['importance_iou'],
                'importance_correlation': coord_sim['importance_correlation'],
                # Descriptor-based
                'embedding_cosine': desc_sim['embedding_cosine'],
                'spatial_hist_corr': desc_sim['spatial_hist_corr'],
                'radial_corr': desc_sim['radial_corr'],
                'quadrant_corr': desc_sim['quadrant_corr'],
                'morans_similarity': desc_sim['morans_similarity'],
                'value_hist_corr': desc_sim['value_hist_corr'],
                # Regional (dynamic)
                'avg_region_score': regional_sim['avg_region_score'],
                'weighted_region_score': regional_sim['weighted_region_score'],
                # Combined
                'coord_component': breakdown['coordinate_component'],
                'desc_component': breakdown['descriptor_component'],
                'regional_component': breakdown['regional_component'],
                'combined_score': combined
            }
            
            # Add individual region scores
            for region_id, score in regional_sim.get('region_scores', {}).items():
                match_dict[f'region_{region_id}_score'] = score
            
            matches.append(match_dict)
        
        return pd.DataFrame(matches).sort_values('combined_score', ascending=False).head(top_k)
    
    def run_analysis(self, epochs=150, top_k=20, clustering_method='kmeans'):
        """Run analysis with pre-computed region clustering"""
        print("\n" + "="*70)
        print("GENE-TO-M/Z MATCHING V8 - PRE-COMPUTED CLUSTERING")
        print(f"RNA: {RNA_PIXEL_SIZE}μm (hexagonal) | MSI: {MSI_PIXEL_SIZE}μm (Cartesian)")
        print(f"Clustering: {clustering_method.upper()}, {N_CLUSTERS} regions")
        print("Hybrid: Coordinate + Descriptor + Regional matching")
        print("="*70)
        
        # =============================================
        # STEP 0: Pre-compute region labels for all samples
        # =============================================
        self.precompute_region_labels(method=clustering_method, n_clusters=N_CLUSTERS)
        
        # Gene availability
        gene_avail = {}
        for gene in AAD_TARGET_GENES:
            gene_avail[gene] = {s: gene in self.rna_data[s].var_names 
                               for s in RNA_SAMPLE_IDS if s in self.rna_data}
        
        print("\nGene availability:")
        for gene in AAD_TARGET_GENES:
            n = sum(gene_avail[gene].values())
            print(f"  {gene}: {n}/{len(RNA_SAMPLE_IDS)}")
        
        # Train
        print("\n" + "-"*70)
        print("PHASE 1: Training GNN Models")
        print("-"*70)
        
        # RNA
        print(f"\nRNA (6 neighbors, {RNA_PIXEL_SIZE}μm)...")
        rna_train, rna_bio = {}, {}
        for sid in list(self.rna_data.keys())[:4]:
            adata = self.rna_data[sid]
            coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
            features = adata.X[:, :50].toarray() if hasattr(adata.X, 'toarray') else adata.X[:, :50]
            data, bio = self.prepare_data(coords, features, 6)
            rna_train[sid] = data
            rna_bio[sid] = bio
            print(f"  {sid}: {data.x.shape}")
        
        self.rna_model = self.train_model(rna_train, rna_bio, 'rna', epochs)
        
        # MSI
        print(f"\nMSI (8 neighbors, {MSI_PIXEL_SIZE}μm)...")
        msi_train, msi_bio = {}, {}
        for sid in list(self.msi_data.keys())[:4]:
            adata = self.msi_data[sid]
            coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
            features = adata.X[:, :50].toarray() if hasattr(adata.X, 'toarray') else adata.X[:, :50]
            data, bio = self.prepare_data(coords, features, 8)
            msi_train[sid] = data
            msi_bio[sid] = bio
            print(f"  {sid}: {data.x.shape}")
        
        self.msi_model = self.train_model(msi_train, msi_bio, 'msi', epochs)
        
        # Match
        print("\n" + "-"*70)
        print("PHASE 2: Matching (using pre-computed regions)")
        print("-"*70)
        
        all_results = []
        
        for gene in AAD_TARGET_GENES:
            print(f"\n{'='*50}")
            print(f"Gene: {gene}")
            print(f"{'='*50}")
            
            available = [s for s, a in gene_avail[gene].items() if a]
            if not available:
                continue
            
            for rna_sample in available:
                print(f"\n  {rna_sample}")
                
                # Get gene data
                adata = self.rna_data[rna_sample]
                rna_coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
                gene_idx = list(adata.var_names).index(gene)
                gene_expr = adata.X[:, gene_idx].toarray().flatten() if hasattr(adata.X, 'toarray') \
                           else adata.X[:, gene_idx].flatten()
                
                # Get MSI coords for alignment
                msi_sample = rna_sample
                if msi_sample not in self.msi_data:
                    print(f"    MSI {msi_sample} not found")
                    continue
                
                msi_adata = self.msi_data[msi_sample]
                msi_coords = np.column_stack([msi_adata.obs['x_um'].values, msi_adata.obs['y_um'].values])
                
                # Get pre-computed region labels
                rna_region_labels = self.rna_region_labels.get(rna_sample)
                msi_region_labels = self.msi_region_labels.get(msi_sample)
                
                # Extract gene signature using pre-computed regions
                print(f"    Extracting gene signature (using pre-computed regions)...")
                gene_sig = self.extract_signature(
                    rna_coords, gene_expr, rna_sample, gene, 'gene',
                    self.rna_model, 6, msi_coords, 
                    region_labels=rna_region_labels,
                    verbose=False
                )
                
                if gene_sig.region_labels is not None:
                    unique, counts = np.unique(gene_sig.region_labels, return_counts=True)
                    size_str = ", ".join([f"R{u}={c}" for u, c in zip(unique, counts)])
                    print(f"    Gene regions: {size_str}")
                
                # Save visualizations
                self.visualize_signature_with_regions(
                    gene_sig,
                    os.path.join(self.output_dir, 'saliency_maps', f'{gene}_{rna_sample}.png')
                )
                self.visualize_signature_with_regions(
                    gene_sig,
                    os.path.join(self.output_dir, 'gene_visualizations', f'{gene}_{rna_sample}.png')
                )
                
                # Save embeddings
                with open(os.path.join(self.output_dir, 'embeddings', f'{gene}_{rna_sample}.pkl'), 'wb') as f:
                    pickle.dump({
                        'global_embedding': gene_sig.global_embedding,
                        'node_importance': gene_sig.node_importance,
                        'morans_i': gene_sig.morans_i,
                        'region_labels': gene_sig.region_labels,
                        'region_descriptors': gene_sig.region_descriptors
                    }, f)
                
                # Extract m/z signatures using pre-computed regions
                print(f"    Matching vs MSI (using pre-computed regions)...")
                msi_data = msi_adata.X.toarray() if hasattr(msi_adata.X, 'toarray') else msi_adata.X
                mz_names = list(msi_adata.var_names)
                
                mz_sigs = {}
                for i, mz_name in enumerate(mz_names):
                    mz_sigs[mz_name] = self.extract_signature(
                        msi_coords, msi_data[:, i], msi_sample, mz_name, 'mz',
                        self.msi_model, 8, 
                        region_labels=msi_region_labels
                    )
                    if (i + 1) % 100 == 0:
                        print(f"      {i+1}/{len(mz_names)}")
                
                # Find matches
                matches = self.find_matches(gene_sig, mz_sigs, top_k)
                all_results.append(matches)
                
                if len(matches) > 0:
                    print(f"\n    Top 5 (with regional scores):")
                    for idx in range(min(5, len(matches))):
                        m = matches.iloc[idx]
                        region_str = ", ".join([f"R{i}={m.get(f'region_{i}_score', 0):.2f}" 
                                               for i in range(N_CLUSTERS)])
                        print(f"      {idx+1}. m/z {m['mz_feature']}: "
                              f"combined={m['combined_score']:.3f} ({region_str})")
                    
                    # Visualize top match
                    top = matches.iloc[0]
                    top_mz = mz_sigs[top['mz_feature']]
                    
                    # Align labels
                    gene_aligned, mz_aligned = self.align_region_labels(gene_sig, top_mz)
                    
                    coord_sim = compute_coordinate_based_similarity(gene_aligned, mz_aligned)
                    desc_sim = compute_descriptor_similarity(gene_aligned, mz_aligned)
                    regional_sim = compute_regional_similarity(gene_aligned, mz_aligned)
                    
                    self.visualize_match_with_regions(
                        gene_aligned, mz_aligned, coord_sim, desc_sim, regional_sim,
                        top['combined_score'],
                        os.path.join(self.output_dir, 'match_visualizations', f'{gene}_{rna_sample}_top.png')
                    )
        
        # Save results
        if all_results:
            results = pd.concat(all_results, ignore_index=True)
            results = results.sort_values('combined_score', ascending=False)
            results.to_csv(os.path.join(self.output_dir, 'gene_to_mz_matches_v8_kmeans.csv'), index=False)
            
            print("\n" + "="*70)
            print("TOP MATCHES (with Regional Analysis)")
            print("="*70)
            
            for gene in AAD_TARGET_GENES:
                gr = results[results['gene'] == gene]
                if len(gr) > 0:
                    t = gr.iloc[0]
                    region_str = ", ".join([f"R{i}={t.get(f'region_{i}_score', 0):.3f}" 
                                           for i in range(N_CLUSTERS)])
                    print(f"\n{gene}:")
                    print(f"  Best: m/z {t['mz_feature']} ({t['rna_sample']})")
                    print(f"  Combined: {t['combined_score']:.4f}")
                    print(f"  Regional: {region_str}")
            
            return results
        
        return None


def main():
    print("="*70)
    print("V8: Pre-computed Clustering + Regional Matching")
    print(f"RNA: {RNA_PIXEL_SIZE}μm | MSI: {MSI_PIXEL_SIZE}μm | N_CLUSTERS: {N_CLUSTERS}")
    print(f"Clustering Method: {CLUSTERING_METHOD}")
    print("="*70)
    
    matcher = HybridMatcherWithKMeans(output_dir='./gene_to_mz_results_v8_kmeans')
    matcher.load_all_data()
    results = matcher.run_analysis(epochs=150, top_k=20, clustering_method=CLUSTERING_METHOD)
    
    print("\n" + "="*70)
    print("COMPLETE!")
    print("="*70)
    
    return matcher, results


if __name__ == "__main__":
    matcher, results = main()

V8: Pre-computed Clustering + Regional Matching
RNA: 55μm | MSI: 60μm | N_CLUSTERS: 3
Clustering Method: leiden
Loading RNA-seq data...
  Pixel size: 55 μm (hexagonal)
  YC_1: (2112, 800)
  YC_2: (2775, 800)
  YC_3: (2808, 800)
  YC_4: (2725, 800)
  YAD_1: (2915, 800)
  YAD_2: (2960, 800)
  YAD_3: (2880, 800)
  YAD_4: (2939, 800)
  AC_1: (3065, 800)
  AC_2: (3054, 800)
  AC_3: (2892, 800)
  AC_4: (3002, 800)
  AAD_1: (2700, 800)
  AAD_2: (2171, 800)
  AAD_3: (1584, 800)
  AAD_4: (2438, 800)

Loading MSI data...
  Pixel size: 60 μm (Cartesian)
  YC_1: (6688, 528)
  YC_2: (7858, 528)
  YC_3: (7150, 528)
  YC_4: (6067, 528)
  YAD_1: (7517, 528)
  YAD_2: (7596, 528)
  YAD_3: (6844, 528)
  YAD_4: (7591, 528)
  AC_1: (6955, 528)
  AC_2: (5729, 528)
  AC_3: (7569, 528)
  AC_4: (7792, 528)
  AAD_1: (6471, 528)
  AAD_2: (5959, 528)
  AAD_3: (5392, 528)
  AAD_4: (6833, 528)

GENE-TO-M/Z MATCHING V8 - PRE-COMPUTED CLUSTERING
RNA: 55μm (hexagonal) | MSI: 60μm (Cartesian)
Clustering: LEIDEN, 3 regi

KeyboardInterrupt: 